# Linear and Logistic Regression (and their Regularized Variants)

This notebook provides a university-level introduction to **Linear Regression** (for continuous targets) and **Logistic Regression** (for binary / multi-class targets), together with their regularized variants (Ridge, Lasso, Elastic Net, L1/L2-penalized logistic regression). For each model we present both the `scikit-learn` implementation (for prediction / pipelines) and the `statsmodels` implementation (for classical statistical inference).

Author: Michał Woźniak

## Technical Setup: Running This Notebook Locally with `uv`

### Prerequisites
- Ensure you have `uv` installed on your system. If not, install it following the instructions at [https://github.com/astral-sh/uv](https://github.com/astral-sh/uv)

### Step 1: Initialize a New Project
Navigate to your working directory and initialize a new Python project:
```bash
# Navigate to your project directory
cd /path/to/your/project

# Initialize uv project (if not already initialized)
uv init

# Or if you want to specify Python version
uv init --python 3.11
```

### Step 2: Install Required Dependencies
```bash
# Install core data science packages
uv add numpy pandas scikit-learn scipy

# Install statsmodels for classical statistical inference
uv add statsmodels

# Install visualization libraries
uv add matplotlib seaborn

# Install Jupyter notebook
uv add jupyter ipykernel

# Alternative: Install all at once
uv add numpy pandas scikit-learn scipy statsmodels matplotlib seaborn jupyter ipykernel
```

### Step 3: Activate the Environment and Start Jupyter
```bash
uv run jupyter notebook
# or
uv run jupyter lab
```

### Step 4: Open This Notebook
Once Jupyter starts, navigate to `linear_and_logistic_regression.ipynb` and open it.

### Required Packages Summary
- **numpy** (≥1.21.0): Numerical computations and array operations
- **pandas** (≥1.3.0): Data manipulation and analysis
- **scikit-learn** (≥1.0.0): Machine learning algorithms and utilities
- **scipy** (≥1.7.0): Scientific computing
- **statsmodels** (≥0.13.0): Statistical modeling and inference (OLS, GLM, t-tests, confidence intervals)
- **matplotlib** (≥3.4.0): Basic plotting and visualization
- **seaborn** (≥0.11.0): Statistical data visualization
- **jupyter**: Jupyter notebook interface
- **ipykernel**: Jupyter kernel for Python

### Troubleshooting
- If you encounter import errors, ensure all packages are installed: `uv pip list`
- To update packages: `uv add --upgrade package_name`
- To see your Python version: `uv run python --version`


# 1. Introduction to Linear and Logistic Regression

## What are Linear and Logistic Regression?

Both are **supervised**, **parametric**, **eager-learning** algorithms that model the target as a function of a **linear combination** of the input features. They differ only in what that linear combination is used for:

| | Linear Regression | Logistic Regression |
|---|---|---|
| **Target** | Continuous (`y ∈ ℝ`) | Binary / categorical (`y ∈ {0,1}` or `{1, …, K}`) |
| **Prediction** | $\hat{y} = x^\top w + b$ | $\hat{p} = \sigma(x^\top w + b)$ |
| **Loss** | Mean Squared Error (MSE) | Binary / categorical cross-entropy |
| **Solver** | Closed-form (normal equation) | Iterative (no closed form) |
| **Probabilistic story** | MLE under Gaussian noise | MLE under Bernoulli / Multinoulli response |

### Key Characteristics

1. **Parametric**: The model is fully described by a fixed number of parameters $w$ (coefficients) and $b$ (intercept). Training data is summarized into these parameters and then discarded.
2. **Interpretable**: Each coefficient $w_j$ has a direct interpretation.
3. **Fast training and prediction**: Training is $O(np^2 + p^3)$ for OLS; logistic regression is iterative but also fast.
4. **Strong baseline**: Any non-linear model should beat these to justify its complexity.

### Mathematical Foundation — Linear Regression

#### Ordinary Least Squares (OLS)

Assume $y_i = x_i^\top w + b + \varepsilon_i$ with $\mathbb{E}[\varepsilon_i] = 0$. OLS chooses $w$ to minimize the **sum of squared residuals**:

$$\mathcal{L}(w) = \frac{1}{2n}\sum_{i=1}^{n}(y_i - x_i^\top w)^2 = \frac{1}{2n}\|y - Xw\|_2^2$$

Setting $\nabla_w \mathcal{L}(w) = 0$ gives the **normal equation**:

$$X^\top X \, w = X^\top y \quad\Longrightarrow\quad \hat{w} = (X^\top X)^{-1} X^\top y$$

provided $X^\top X$ is invertible (i.e. the columns of $X$ are linearly independent).

#### Geometric Interpretation
$X\hat{w}$ is the **orthogonal projection** of $y$ onto the column space of $X$. The residual vector $y - X\hat{w}$ is orthogonal to every feature column.

#### Probabilistic Interpretation (MLE)
If $y_i\mid x_i \sim \mathcal{N}(x_i^\top w,\sigma^2)$ then maximizing the log-likelihood is equivalent to minimizing the sum of squared errors — so **OLS = MLE under Gaussian noise**.

#### Gauss–Markov Theorem
Under linearity, exogeneity ($\mathbb{E}[\varepsilon\mid X]=0$), homoskedasticity and no perfect multicollinearity, OLS is the **Best Linear Unbiased Estimator (BLUE)**: smallest variance among linear unbiased estimators.

---

### Mathematical Foundation — Logistic Regression

#### The Sigmoid and the Log-Odds

For binary classification with $y_i \in \{0,1\}$ we use the **logistic (sigmoid) function**:

$$\sigma(z) = \frac{1}{1 + e^{-z}} \in (0,1)$$

The model is

$$\mathbb{P}(y=1\mid x) = \sigma(x^\top w + b)$$

Equivalently, the log-odds ("logit") is linear in $x$:

$$\log\frac{\mathbb{P}(y=1\mid x)}{\mathbb{P}(y=0\mid x)} = x^\top w + b$$

So a coefficient $w_j$ means: “a one-unit increase in $x_j$ (holding others fixed) increases the log-odds of $y=1$ by $w_j$”, or equivalently multiplies the odds by $e^{w_j}$.

#### Cross-Entropy Loss

The negative log-likelihood under a Bernoulli model gives:

$$\mathcal{L}(w,b) = -\frac{1}{n}\sum_{i=1}^n \Bigl[y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\Bigr]$$

where $\hat{p}_i = \sigma(x_i^\top w + b)$. The gradient is remarkably clean:

$$\nabla_w \mathcal{L} = \frac{1}{n}\sum_i (\hat{p}_i - y_i)\, x_i = \frac{1}{n} X^\top (\hat{p} - y)$$
$$\nabla_b \mathcal{L} = \frac{1}{n}\sum_i (\hat{p}_i - y_i)$$

#### Why No Closed Form?
The cross-entropy loss is **convex** (so it has a unique global minimum) but involves $\sigma(\cdot)$, so there is no closed-form solution. In practice we solve it iteratively with:
- **Gradient descent / SGD** (what we’ll implement from scratch)
- **Newton’s method** → known as **Iteratively Reweighted Least Squares (IRLS)** in this context, and what `statsmodels` and old-school GLM packages use
- **L-BFGS / liblinear / SAGA** — the defaults in `sklearn`

#### Multiclass Extension: Softmax Regression
For $K$ classes we generalize to

$$\mathbb{P}(y=k\mid x) = \frac{e^{x^\top w_k + b_k}}{\sum_{j=1}^K e^{x^\top w_j + b_j}}$$

This is also called **multinomial logistic regression** or **softmax regression**.

---

### Why Regularization?

Both models suffer from the same two failure modes:

1. **Multicollinearity**: correlated features lead to unstable coefficients with huge variance.
2. **High-dimensional or small-sample problems**: when $p \gtrsim n$, the unregularized problem is ill-posed and severely overfits.

**Regularization** fixes this by adding a penalty on the size of $w$, trading a little bias for a lot less variance.

| Variant | Penalty added to loss | Effect |
|---|---|---|
| **Ridge (L2)** | $\tfrac{\alpha}{2}\|w\|_2^2$ | smooth shrinkage; stabilizes correlated features |
| **Lasso (L1)** | $\alpha \|w\|_1$ | drives coefficients exactly to zero → variable selection |
| **Elastic Net** | $\alpha\rho\|w\|_1 + \tfrac{\alpha(1-\rho)}{2}\|w\|_2^2$ | both at once |

Regularized logistic regression works identically — just add the penalty term to the cross-entropy loss.

**`sklearn` quirk**: for logistic regression, `sklearn` parameterizes regularization strength as $C = 1/\alpha$ (larger $C$ → less regularization). This matches the SVM convention.


## Key Hyperparameters

### 1. Regularization Strength ($\alpha$ or $C = 1/\alpha$)
- **$\alpha = 0$ / $C = \infty$**: pure OLS / unpenalized logistic — minimum bias, maximum variance.
- **Large $\alpha$ / small $C$**: heavy shrinkage — coefficients close to zero, high bias but low variance.
- **Best practice**: always search on a **logarithmic scale** (e.g. $10^{-4}$ to $10^{4}$).

### 2. L1 Ratio ($\rho$, Elastic Net)
- $\rho = 0$: pure L2 (Ridge). $\rho = 1$: pure L1 (Lasso). Mixture in between.
- Start with $\rho = 0.5$ and tune with CV.

### 3. `fit_intercept`
- Almost always keep it `True`. Setting it `False` forces the line through the origin — rarely what you want.

### 4. `class_weight` (Logistic only)
- Use `class_weight='balanced'` for imbalanced data.

## Advantages and Limitations

### Advantages ✓
1. **Simple and interpretable** — each coefficient has a direct meaning.
2. **Convex optimization** — unique global optimum for OLS (closed form) and logistic (iterative but convex).
3. **Statistical inference** — `statsmodels` gives p-values, CIs, hypothesis tests.
4. **Fast** — training and prediction scale very well with $n$ and $p$.
5. **Excellent baseline** — any non-linear model should beat these to justify its complexity.

### Limitations ✗
1. **Linearity assumption** — miss non-linear relationships unless you feature-engineer them.
2. **Sensitive to outliers** — squared loss heavily penalizes large residuals; logistic is milder but still pulled by extremes.
3. **Multicollinearity blows up variance** — unless regularized.
4. **Require feature scaling for regularized variants** — the L1/L2 penalty treats every coefficient the same regardless of the feature scale.
5. **Decision boundary is a hyperplane (logistic)** — no curvature without feature engineering.

## Critical Preprocessing: Feature Scaling

**For plain OLS**, scaling is *not* mathematically required — the solution is invariant to linear rescaling of individual features.

**For Ridge / Lasso / Elastic Net** (both linear and logistic), scaling is **essential**. Consider two features:
- Feature 1: Age (range 20–80)
- Feature 2: Income (range 20,000–200,000)

The penalty $\alpha\|w\|_2^2$ penalizes `w_income` much less than `w_age` because a small `w_income` already produces a large contribution to the prediction. After standardization the penalty applies fairly to all features.

**Common scaling methods:**

1. **Standardization (Z-score normalization)**: zero mean, unit variance
   $$x' = \frac{x - \mu}{\sigma}$$
2. **Min-Max Scaling (Normalization)**: to $[0,1]$
   $$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
3. **Robust Scaling**: median and IQR (outlier-safe)
   $$x' = \frac{x - \text{median}}{\text{IQR}}$$

**Best practice**: always fit the scaler on the training set only, then transform both train and test.

# 2. Linear Regression: From-Scratch Implementation

In this section we implement linear regression from scratch to understand the mechanics. We build:
1. A **closed-form OLS** estimator using the normal equation.
2. A **gradient-descent OLS** estimator that optimizes the MSE iteratively.

Then we compare their behavior on the California Housing dataset.

In [ ]:
import warnings

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Import necessary libraries
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Matplotlib configuration for better plots
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
# Load the California Housing dataset (our regression benchmark)
from sklearn.datasets import fetch_california_housing
from sklearn.utils import resample

housing = fetch_california_housing()
# Subsample to keep from-scratch experiments snappy
X_reg, y_reg = resample(housing.data, housing.target, n_samples=5000, random_state=42)

print("California Housing Dataset Loaded")
print(f"Features shape: {X_reg.shape}")
print(f"Target shape:   {y_reg.shape}")
print(f"\nFeature names: {housing.feature_names}")
print("Target: Median house value (in $100,000s)")
print(f"\nFirst 5 samples:\n{X_reg[:5]}")
print(f"First 5 targets: {y_reg[:5]}")

## 2.1 Closed-Form OLS (Normal Equation)

Recall the normal equation:
$$\hat{w} = (X^\top X)^{-1} X^\top y$$

In practice we never actually invert $X^\top X$ — we solve the linear system with `np.linalg.solve` for numerical stability.

In [ ]:
class OLSRegressorClosedForm:
    """
    Ordinary Least Squares regression using the normal equation.

    Solves (X^T X) w = X^T y directly with np.linalg.solve
    (more numerically stable than computing the inverse).
    """

    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept

    def _add_intercept(self, X):
        if self.fit_intercept:
            return np.hstack([np.ones((X.shape[0], 1)), X])
        return X

    def fit(self, X_train, y_train):
        X_train = np.asarray(X_train, dtype=float)
        y_train = np.asarray(y_train, dtype=float)

        X_ = self._add_intercept(X_train)
        XtX = X_.T @ X_
        Xty = X_.T @ y_train
        theta = np.linalg.solve(XtX, Xty)

        if self.fit_intercept:
            self.intercept_ = theta[0]
            self.coef_ = theta[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = theta
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.coef_ + self.intercept_

    def score(self, X, y):
        """Return the coefficient of determination R^2."""
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot


print("OLSRegressorClosedForm class created successfully!")

### Split data and (optionally) scale features

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

print(f"Training set size: {X_train_r.shape[0]}")
print(f"Test set size:     {X_test_r.shape[0]}")

# For plain OLS, scaling is not strictly required, but it makes
# coefficient magnitudes comparable and is REQUIRED for the
# regularized variants later.
scaler_r = StandardScaler()
scaler_r.fit(X_train_r)
X_train_rs = scaler_r.transform(X_train_r)
X_test_rs = scaler_r.transform(X_test_r)

print("\nBefore scaling (first 3 training samples):")
print(X_train_r[:3])
print("\nAfter scaling (first 3 training samples):")
print(X_train_rs[:3])

### Train the closed-form OLS model

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ols_cf = OLSRegressorClosedForm(fit_intercept=True)

start_time = time.time()
ols_cf.fit(X_train_rs, y_train_r)
cf_train_time = time.time() - start_time

start_time = time.time()
y_pred_cf = ols_cf.predict(X_test_rs)
cf_pred_time = time.time() - start_time

r2_cf = r2_score(y_test_r, y_pred_cf)
rmse_cf = np.sqrt(mean_squared_error(y_test_r, y_pred_cf))
mae_cf = mean_absolute_error(y_test_r, y_pred_cf)

print("=" * 60)
print("CLOSED-FORM OLS RESULTS")
print("=" * 60)
print(f"Training time:   {cf_train_time:.6f} s")
print(f"Prediction time: {cf_pred_time:.6f} s")
print(f"R^2 score:       {r2_cf:.4f}")
print(f"RMSE:            {rmse_cf:.4f}")
print(f"MAE:             {mae_cf:.4f}")

print(f"\nIntercept: {ols_cf.intercept_:.4f}")
print("Coefficients (on scaled features):")
for name, w in zip(housing.feature_names, ols_cf.coef_):
    print(f"  {name:<10} {w:+.4f}")

## 2.2 Gradient-Descent OLS

### Why bother with gradient descent?

The closed-form solution is beautiful but has costs:
- Building and solving $(X^\top X)\,w = X^\top y$ is $O(np^2 + p^3)$ and requires $O(p^2)$ memory.
- For very large $p$ (or streaming data), this is infeasible.

**Gradient descent** iteratively updates $w$ using the gradient of the MSE:

$$\mathcal{L}(w,b) = \frac{1}{2n}\sum_i (y_i - x_i^\top w - b)^2$$
$$\nabla_w \mathcal{L} = -\tfrac{1}{n} X^\top (y - Xw - b)$$
$$\nabla_b \mathcal{L} = -\tfrac{1}{n}\sum_i (y_i - x_i^\top w - b)$$

Each step costs $O(np)$ — linear in both dimensions — at the cost of tuning a learning rate and a convergence criterion.

In [ ]:
class OLSRegressorGD:
    """
    Ordinary Least Squares regression trained with (batch) gradient descent
    on the mean-squared-error objective.
    """

    def __init__(self, learning_rate=0.01, n_iters=1000, tol=1e-6):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.tol = tol

    def fit(self, X_train, y_train):
        X_train = np.asarray(X_train, dtype=float)
        y_train = np.asarray(y_train, dtype=float)
        n, p = X_train.shape

        self.coef_ = np.zeros(p)
        self.intercept_ = 0.0
        self.loss_history_ = []

        for it in range(self.n_iters):
            y_pred = X_train @ self.coef_ + self.intercept_
            resid = y_train - y_pred

            grad_w = -(X_train.T @ resid) / n
            grad_b = -resid.mean()

            self.coef_ -= self.learning_rate * grad_w
            self.intercept_ -= self.learning_rate * grad_b

            if it % 10 == 0:
                loss = 0.5 * np.mean(resid**2)
                self.loss_history_.append(loss)
                if len(self.loss_history_) > 2 and abs(self.loss_history_[-2] - self.loss_history_[-1]) < self.tol:
                    break
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_

    def score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1.0 - ss_res / ss_tot


print("OLSRegressorGD class created successfully!")

### Performance comparison: closed-form vs gradient descent

In [ ]:
ols_gd = OLSRegressorGD(learning_rate=0.05, n_iters=2000, tol=1e-8)

start_time = time.time()
ols_gd.fit(X_train_rs, y_train_r)
gd_train_time = time.time() - start_time

start_time = time.time()
y_pred_gd = ols_gd.predict(X_test_rs)
gd_pred_time = time.time() - start_time

r2_gd = r2_score(y_test_r, y_pred_gd)

print("=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)
print(f"\n{'Method':<22} {'Train Time (s)':<15} {'Predict Time (s)':<18} {'R^2':<10}")
print("-" * 60)
print(f"{'Closed-form OLS':<22} {cf_train_time:<15.6f} {cf_pred_time:<18.6f} {r2_cf:<10.4f}")
print(f"{'Gradient-descent OLS':<22} {gd_train_time:<15.6f} {gd_pred_time:<18.6f} {r2_gd:<10.4f}")
print("-" * 60)

plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(ols_gd.loss_history_)) * 10, ols_gd.loss_history_, "o-", markersize=3)
plt.xlabel("Iteration")
plt.ylabel("MSE / 2")
plt.title("Gradient-Descent OLS - Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nCoefficient agreement (max |diff|):", f"{np.max(np.abs(ols_cf.coef_ - ols_gd.coef_)):.6f}")
print("Both methods converge to the same solution.")

### Visualizing the fit: Predicted vs Actual and Residual plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_test_r, y_pred_cf, alpha=0.4, s=20)
lims = [y_test_r.min(), y_test_r.max()]
axes[0].plot(lims, lims, "r--", lw=2, label="Perfect Prediction")
axes[0].set_xlabel("Actual ($100k)")
axes[0].set_ylabel("Predicted ($100k)")
axes[0].set_title("Closed-form OLS: Predicted vs Actual", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

residuals_cf = y_test_r - y_pred_cf
axes[1].scatter(y_pred_cf, residuals_cf, alpha=0.4, s=20)
axes[1].axhline(0, color="r", linestyle="--", lw=2)
axes[1].set_xlabel("Predicted ($100k)")
axes[1].set_ylabel("Residuals")
axes[1].set_title("Residual Plot", fontweight="bold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Good residual plots show:")
print("1. Random scatter around zero (no pattern)")
print("2. Constant variance across all predicted values (homoskedasticity)")
print("3. Most points close to the zero line")

# 3. Linear Regression: Scikit-learn Implementation

Now that we understand how Linear Regression works internally, let's use scikit-learn's optimized implementation. `LinearRegression` uses a numerically stable SVD-based solver under the hood, and exposes a clean, consistent API shared across all regression variants.

In [ ]:
from sklearn.linear_model import LinearRegression

lr_sklearn = LinearRegression(fit_intercept=True)
lr_sklearn.fit(X_train_rs, y_train_r)

y_pred_sklearn = lr_sklearn.predict(X_test_rs)

r2 = r2_score(y_test_r, y_pred_sklearn)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_sklearn))
mae = mean_absolute_error(y_test_r, y_pred_sklearn)

print("=" * 60)
print("SCIKIT-LEARN LINEAR REGRESSION RESULTS")
print("=" * 60)
print(f"R^2 score: {r2:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"MAE:       {mae:.4f}")

print(f"\nIntercept: {lr_sklearn.intercept_:.4f}")
coef_df = pd.DataFrame({"feature": housing.feature_names, "coef": lr_sklearn.coef_})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)
print("Coefficients (on scaled features):")
print(coef_df[["feature", "coef"]].to_string(index=False))

print(f"\nMax |sklearn.coef - our closed-form.coef|: {np.max(np.abs(lr_sklearn.coef_ - ols_cf.coef_)):.2e}")

In [ ]:
# Visualize coefficients
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["tab:blue" if c > 0 else "tab:red" for c in coef_df["coef"]]
ax.barh(coef_df["feature"], coef_df["coef"], color=colors, edgecolor="black")
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Coefficient (standardized features)")
ax.set_title("Linear Regression Coefficients", fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print("\nInterpretation (standardized features):")
print("A coefficient of +0.5 means: 'a 1-std increase in this feature is")
print("associated with a +0.5 ($50k) increase in predicted price,")
print("holding all other features fixed'.")

# 4. Linear Regression: Statsmodels Implementation

Scikit-learn is wonderful for machine learning — predictions, pipelines, cross-validation — but it deliberately does not expose classical **statistical inference** quantities: standard errors, t-statistics, p-values, confidence intervals, adjusted $R^2$, F-tests, etc.

For that, we switch to **statsmodels**, which is designed around the econometric / statistical tradition.

## 4.1 OLS via `statsmodels.api.OLS`

Two important differences from sklearn:

1. You must **manually add a constant (intercept) column** using `sm.add_constant(X)`.
2. The argument order is **`OLS(y, X)`**, not `fit(X, y)`.

In [ ]:
import statsmodels.api as sm

# Add intercept column (statsmodels does NOT do this automatically)
X_train_sm = sm.add_constant(X_train_rs)
X_test_sm = sm.add_constant(X_test_rs)

# Note the argument order: y first, X second
ols_sm = sm.OLS(y_train_r, X_train_sm)
results_sm = ols_sm.fit()

# Full regression summary
print(results_sm.summary(xname=["const"] + list(housing.feature_names)))

In [ ]:
# Individual pieces of the summary are easy to access programmatically
summary_df = pd.DataFrame(
    {
        "coef": results_sm.params,
        "std_err": results_sm.bse,
        "t": results_sm.tvalues,
        "p>|t|": results_sm.pvalues,
        "CI_low": results_sm.conf_int()[:, 0],
        "CI_high": results_sm.conf_int()[:, 1],
    },
    index=["const"] + list(housing.feature_names),
)
print("Coefficient table:")
print(summary_df.round(4))

print(f"\nR^2:            {results_sm.rsquared:.4f}")
print(f"Adjusted R^2:   {results_sm.rsquared_adj:.4f}")
print(f"F-statistic:    {results_sm.fvalue:.2f}")
print(f"Prob (F-stat):  {results_sm.f_pvalue:.2e}")
print(f"AIC:            {results_sm.aic:.2f}")
print(f"BIC:            {results_sm.bic:.2f}")

print("\nSanity check: sklearn vs statsmodels coefficients should match:")
print(f"  Max |diff|           = {np.max(np.abs(results_sm.params[1:] - lr_sklearn.coef_)):.2e}")
print(f"  |diff| for intercept = {abs(results_sm.params[0] - lr_sklearn.intercept_):.2e}")

### Interpreting the statsmodels output

**Coefficient table:**
* `coef`: Point estimate $\hat{w}_j$.
* `std err`: Standard error of $\hat{w}_j$ — how uncertain is the estimate?
* `t`: $\hat{w}_j / \text{SE}(\hat{w}_j)$, the test statistic for $H_0: w_j = 0$.
* `P>|t|`: Two-sided p-value. Small ($<0.05$) → the feature is statistically significant.
* `[0.025, 0.975]`: 95% confidence interval for $w_j$.

**Model-level diagnostics:**
* **R-squared / Adj. R-squared**: variance explained; adjusted version penalizes extra features.
* **F-statistic / Prob(F)**: tests whether *all* coefficients are jointly zero.
* **AIC / BIC**: information criteria for model comparison — lower is better.
* **Durbin-Watson**: tests autocorrelation of residuals (important for time series).
* **Jarque-Bera / Prob(JB)**: tests normality of residuals.
* **Condition number**: if very large (>30), suggests multicollinearity.

Use statsmodels when you care about **inference** — “is this coefficient significantly different from zero?” — rather than pure prediction.

In [ ]:
# Diagnostic plots: residuals
y_pred_sm = results_sm.predict(X_test_sm)
residuals_sm = y_test_r - y_pred_sm

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_pred_sm, residuals_sm, alpha=0.4, s=20)
axes[0].axhline(0, color="r", linestyle="--", lw=2)
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Fitted", fontweight="bold")
axes[0].grid(True, alpha=0.3)

from scipy import stats as scipy_stats

scipy_stats.probplot(residuals_sm, dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q Plot of Residuals", fontweight="bold")
axes[1].grid(True, alpha=0.3)

axes[2].hist(residuals_sm, bins=40, edgecolor="black", alpha=0.7)
axes[2].axvline(0, color="r", linestyle="--", lw=2)
axes[2].set_xlabel("Residuals")
axes[2].set_ylabel("Frequency")
axes[2].set_title("Residual Distribution", fontweight="bold")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4.2 The Formula API (R-style)

Statsmodels also supports an **R-style formula interface** via `statsmodels.formula.api`. This is often the most readable way to fit a model — and it handles categorical variables and interactions gracefully.

In [ ]:
import statsmodels.formula.api as smf

train_df = pd.DataFrame(X_train_rs, columns=housing.feature_names)
train_df["MedHouseVal"] = y_train_r

formula = "MedHouseVal ~ " + " + ".join(housing.feature_names)
print(f"Formula: {formula}\n")

ols_formula = smf.ols(formula=formula, data=train_df).fit()
print(ols_formula.summary().tables[1])

# 5. Regularized Linear Regression

We've seen plain OLS. Now let's explore the three most common regularized variants: **Ridge** (L2), **Lasso** (L1), and **Elastic Net** (L1 + L2).

To make the differences between these methods visible, we'll work on a version of the problem with many correlated / redundant features — that is where regularization really shines.

## 5.1 Building a Challenging Dataset

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Expand the feature set with polynomial interactions so we get many correlated features
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_train_poly = poly.fit_transform(X_train_r)
X_test_poly = poly.transform(X_test_r)

scaler_poly = StandardScaler()
X_train_poly_s = scaler_poly.fit_transform(X_train_poly)
X_test_poly_s = scaler_poly.transform(X_test_poly)

poly_feature_names = poly.get_feature_names_out(housing.feature_names)

print(f"Original feature count:   {X_train_r.shape[1]}")
print(f"Polynomial feature count: {X_train_poly.shape[1]}")

corr = np.corrcoef(X_train_poly_s, rowvar=False)
np.fill_diagonal(corr, 0)
print(f"Mean |corr| between features: {np.mean(np.abs(corr)):.3f}")
print(f"Max  |corr| between features: {np.max(np.abs(corr)):.3f}")

## 5.2 Ridge Regression (L2)

$$\hat{w}_{\text{ridge}} = \arg\min_w \; \tfrac{1}{2n}\|y - Xw\|_2^2 + \tfrac{\alpha}{2}\|w\|_2^2$$

Ridge shrinks coefficients **smoothly** toward zero but does not set them exactly to zero. Closed form:
$$\hat{w} = (X^\top X + n\alpha I)^{-1} X^\top y$$

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_poly_s, y_train_r)
y_pred_ridge = ridge.predict(X_test_poly_s)

print("=" * 60)
print("RIDGE RESULTS (alpha=1.0)")
print("=" * 60)
print(f"R^2:  {r2_score(y_test_r, y_pred_ridge):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_ridge)):.4f}")
print(f"\nNon-zero coefficients: {(ridge.coef_ != 0).sum()} / {len(ridge.coef_)}")
print(f"Max |coef|: {np.max(np.abs(ridge.coef_)):.4f}")

In [ ]:
# Sweep alpha on a log scale
alphas = np.logspace(-4, 4, 40)
train_ridge, test_ridge, paths_ridge = [], [], []
for a in alphas:
    r = Ridge(alpha=a)
    r.fit(X_train_poly_s, y_train_r)
    train_ridge.append(r.score(X_train_poly_s, y_train_r))
    test_ridge.append(r.score(X_test_poly_s, y_test_r))
    paths_ridge.append(r.coef_)
paths_ridge = np.array(paths_ridge)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].semilogx(alphas, train_ridge, "o-", label="Train R^2", markersize=4)
axes[0].semilogx(alphas, test_ridge, "s-", label="Test R^2", markersize=4)
axes[0].set_xlabel("alpha (log scale)")
axes[0].set_ylabel("R^2")
axes[0].set_title("Ridge: R^2 vs alpha", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for j in range(paths_ridge.shape[1]):
    axes[1].semilogx(alphas, paths_ridge[:, j], alpha=0.5, lw=1)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("alpha (log scale)")
axes[1].set_ylabel("Coefficient value")
axes[1].set_title("Ridge Coefficient Path", fontweight="bold")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_alpha_ridge = alphas[int(np.argmax(test_ridge))]
print(f"Best alpha: {best_alpha_ridge:.4f}   Best test R^2: {max(test_ridge):.4f}")

## 5.3 Lasso Regression (L1)

$$\hat{w}_{\text{lasso}} = \arg\min_w \; \tfrac{1}{2n}\|y - Xw\|_2^2 + \alpha\|w\|_1$$

Because of the L1 penalty, Lasso drives many coefficients **exactly to zero** — automatic **feature selection**.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01, max_iter=20000)
lasso.fit(X_train_poly_s, y_train_r)
y_pred_lasso = lasso.predict(X_test_poly_s)

n_nonzero = (lasso.coef_ != 0).sum()
print("=" * 60)
print("LASSO RESULTS (alpha=0.01)")
print("=" * 60)
print(f"R^2:  {r2_score(y_test_r, y_pred_lasso):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_lasso)):.4f}")
print(
    f"\nNon-zero coefficients: {n_nonzero} / {len(lasso.coef_)} (= {n_nonzero / len(lasso.coef_) * 100:.0f}% selected)"
)

lasso_df = pd.DataFrame({"feature": poly_feature_names, "coef": lasso.coef_})
lasso_df = lasso_df[lasso_df["coef"] != 0].copy()
lasso_df["abs_coef"] = lasso_df["coef"].abs()
lasso_df = lasso_df.sort_values("abs_coef", ascending=False).head(10)
print("\nTop selected features:")
print(lasso_df[["feature", "coef"]].to_string(index=False))

In [ ]:
alphas_l = np.logspace(-4, 1, 40)
train_l, test_l, paths_l, sparsity = [], [], [], []
for a in alphas_l:
    la = Lasso(alpha=a, max_iter=20000)
    la.fit(X_train_poly_s, y_train_r)
    train_l.append(la.score(X_train_poly_s, y_train_r))
    test_l.append(la.score(X_test_poly_s, y_test_r))
    paths_l.append(la.coef_)
    sparsity.append((la.coef_ != 0).sum())
paths_l = np.array(paths_l)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].semilogx(alphas_l, train_l, "o-", label="Train R^2", markersize=4)
axes[0].semilogx(alphas_l, test_l, "s-", label="Test R^2", markersize=4)
axes[0].set_xlabel("alpha (log scale)")
axes[0].set_ylabel("R^2")
axes[0].set_title("Lasso: R^2 vs alpha", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for j in range(paths_l.shape[1]):
    axes[1].semilogx(alphas_l, paths_l[:, j], alpha=0.5, lw=1)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("alpha (log scale)")
axes[1].set_ylabel("Coefficient value")
axes[1].set_title("Lasso Coefficient Path", fontweight="bold")
axes[1].grid(True, alpha=0.3)

axes[2].semilogx(alphas_l, sparsity, "o-", markersize=4, color="tab:green")
axes[2].set_xlabel("alpha (log scale)")
axes[2].set_ylabel("# non-zero coefficients")
axes[2].set_title("Lasso Sparsity", fontweight="bold")
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_alpha_lasso = alphas_l[int(np.argmax(test_l))]
print(f"Best alpha: {best_alpha_lasso:.4f}   Best test R^2: {max(test_l):.4f}")
print(f"# features selected at best alpha: {sparsity[int(np.argmax(test_l))]}")

## 5.4 Elastic Net (L1 + L2)

$$\hat{w}_{\text{enet}} = \arg\min_w \; \tfrac{1}{2n}\|y - Xw\|_2^2 + \alpha\rho\|w\|_1 + \tfrac{\alpha(1-\rho)}{2}\|w\|_2^2$$

Elastic Net gets the best of both worlds:
- The L1 part gives **sparsity** (feature selection).
- The L2 part gives **stability** when features are correlated.

In sklearn, `l1_ratio` is our $\rho$.

In [ ]:
from sklearn.linear_model import ElasticNet

enet = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000)
enet.fit(X_train_poly_s, y_train_r)
y_pred_enet = enet.predict(X_test_poly_s)

print("=" * 60)
print("ELASTIC NET RESULTS (alpha=0.01, l1_ratio=0.5)")
print("=" * 60)
print(f"R^2:  {r2_score(y_test_r, y_pred_enet):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_enet)):.4f}")
print(f"\nNon-zero coefficients: {(enet.coef_ != 0).sum()} / {len(enet.coef_)}")

## 5.5 Side-by-Side Comparison: OLS vs Ridge vs Lasso vs Elastic Net

In [ ]:
models_lin = {
    "OLS": LinearRegression(),
    "Ridge (alpha=1)": Ridge(alpha=1.0),
    "Lasso (alpha=0.01)": Lasso(alpha=0.01, max_iter=20000),
    "Elastic Net (alpha=0.01, rho=0.5)": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000),
}

rows = []
coef_matrix = {}
for name, model in models_lin.items():
    model.fit(X_train_poly_s, y_train_r)
    y_pred = model.predict(X_test_poly_s)
    rows.append(
        {
            "model": name,
            "train_R2": model.score(X_train_poly_s, y_train_r),
            "test_R2": model.score(X_test_poly_s, y_test_r),
            "test_RMSE": np.sqrt(mean_squared_error(y_test_r, y_pred)),
            "n_nonzero": int((model.coef_ != 0).sum()),
            "max_|coef|": np.max(np.abs(model.coef_)),
        }
    )
    coef_matrix[name] = model.coef_

print("=" * 90)
print("MODEL COMPARISON ON POLYNOMIAL-EXPANDED FEATURES")
print("=" * 90)
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 7), sharey=True)
for ax, (name, vec) in zip(axes, coef_matrix.items()):
    colors = ["tab:blue" if v > 0 else "tab:red" for v in vec]
    ax.barh(range(len(vec)), vec, color=colors, edgecolor="black", alpha=0.8)
    ax.axvline(0, color="black", lw=1)
    ax.set_title(name, fontweight="bold", fontsize=11)
    ax.grid(True, alpha=0.3, axis="x")
axes[0].set_ylabel("Feature index")
plt.suptitle("Coefficients by Model (polynomial features)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Observation:")
print("- OLS produces large, sometimes wild coefficients because of multicollinearity.")
print("- Ridge shrinks all coefficients smoothly toward zero.")
print("- Lasso zeroes out most coefficients entirely (sparsity).")
print("- Elastic Net is a compromise: some zeros, some mild shrinkage.")

## 5.6 Regularized Linear Regression via Statsmodels

Statsmodels supports Ridge, Lasso, and Elastic Net through the `fit_regularized` method on `OLS`:
- `alpha` — same meaning as in sklearn.
- `L1_wt` — what sklearn calls `l1_ratio` ($\rho$). `L1_wt=0` → Ridge, `L1_wt=1` → Lasso.

Note: for regularized fits, statsmodels does NOT return the classical standard errors / p-values — the sampling distribution of penalized estimators is non-standard. You only get point estimates.

In [ ]:
X_train_poly_sm = sm.add_constant(X_train_poly_s)
X_test_poly_sm = sm.add_constant(X_test_poly_s)

ridge_sm = sm.OLS(y_train_r, X_train_poly_sm).fit_regularized(alpha=1.0, L1_wt=0.0)
lasso_sm = sm.OLS(y_train_r, X_train_poly_sm).fit_regularized(alpha=0.01, L1_wt=1.0)
enet_sm = sm.OLS(y_train_r, X_train_poly_sm).fit_regularized(alpha=0.01, L1_wt=0.5)

print("Comparing statsmodels vs sklearn coefficients on the polynomial features")
print("-" * 70)
print(f"Ridge:       max |diff| = {np.max(np.abs(ridge_sm.params[1:] - models_lin['Ridge (alpha=1)'].coef_)):.2e}")
print(
    f"Lasso:       corr(sklearn, sm) = "
    f"{np.corrcoef(lasso_sm.params[1:], models_lin['Lasso (alpha=0.01)'].coef_)[0, 1]:.4f}"
)
print(
    f"Elastic Net: corr(sklearn, sm) = "
    f"{np.corrcoef(enet_sm.params[1:], models_lin['Elastic Net (alpha=0.01, rho=0.5)'].coef_)[0, 1]:.4f}"
)

print("\nPredictive performance on the test set:")
for name, fit in [("Ridge (sm)", ridge_sm), ("Lasso (sm)", lasso_sm), ("Elastic Net (sm)", enet_sm)]:
    y_pred_s = fit.predict(X_test_poly_sm)
    print(
        f"  {name:<18}  R^2 = {r2_score(y_test_r, y_pred_s):.4f}   "
        f"RMSE = {np.sqrt(mean_squared_error(y_test_r, y_pred_s)):.4f}"
    )

# 6. Logistic Regression: From-Scratch Implementation

We now turn to **logistic regression** — the classification counterpart of linear regression. We'll implement binary logistic regression from scratch using gradient descent on the cross-entropy loss.

Recall:
$$\hat{p}_i = \sigma(x_i^\top w + b) = \frac{1}{1 + e^{-(x_i^\top w + b)}}$$
$$\mathcal{L}(w,b) = -\frac{1}{n}\sum_i\bigl[y_i\log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\bigr]$$
$$\nabla_w \mathcal{L} = \tfrac{1}{n} X^\top(\hat{p} - y),\quad \nabla_b\mathcal{L} = \tfrac{1}{n}\sum_i(\hat{p}_i - y_i)$$

In [ ]:
# Load the Breast Cancer dataset (our classification benchmark)
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X_clf = cancer.data
y_clf = cancer.target  # 0 = malignant, 1 = benign

print("Breast Cancer Wisconsin Dataset Loaded")
print(f"Features shape: {X_clf.shape}")
print(f"Target shape:   {y_clf.shape}")
print(f"\nClasses: {cancer.target_names}")
print(f"Class distribution: {np.bincount(y_clf)}")
print(f"\nFirst 3 feature names: {cancer.feature_names[:3]}")

### Step 1: Implement sigmoid, loss and gradient

In [ ]:
def sigmoid(z):
    """Numerically stable sigmoid."""
    # For large negative z, exp(-z) overflows; we split the logic to
    # avoid warnings and return a stable result.
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    out[~pos] = np.exp(z[~pos]) / (1.0 + np.exp(z[~pos]))
    return out


def cross_entropy(w, b, X, y):
    """Mean binary cross-entropy loss."""
    p = sigmoid(X @ w + b)
    eps = 1e-12  # avoid log(0)
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))


def cross_entropy_gradient(w, b, X, y):
    """Gradient of the mean BCE loss w.r.t. w and b."""
    p = sigmoid(X @ w + b)
    err = p - y
    grad_w = (X.T @ err) / len(y)
    grad_b = err.mean()
    return grad_w, grad_b


# Quick sanity check
w_t = np.array([0.5, -0.2])
b_t = 0.1
X_t = np.array([[1.0, 2.0], [-1.0, -1.0], [0.5, 0.8]])
y_t = np.array([1, 0, 1])
print(f"Loss:   {cross_entropy(w_t, b_t, X_t, y_t):.4f}")
gw, gb = cross_entropy_gradient(w_t, b_t, X_t, y_t)
print(f"grad_w: {gw}")
print(f"grad_b: {gb:.4f}")

### Step 2: Implement the Logistic Regression Classifier from Scratch

In [ ]:
class LogisticRegressionScratch:
    """
    Binary logistic regression trained with (batch) gradient descent on the
    cross-entropy loss. Labels must be in {0, 1}.
    """

    def __init__(self, learning_rate=0.1, n_iters=2000, tol=1e-6):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.tol = tol

    def fit(self, X_train, y_train):
        X_train = np.asarray(X_train, dtype=float)
        y_train = np.asarray(y_train, dtype=float)
        n, p = X_train.shape

        self.coef_ = np.zeros(p)
        self.intercept_ = 0.0
        self.loss_history_ = []

        for it in range(self.n_iters):
            gw, gb = cross_entropy_gradient(self.coef_, self.intercept_, X_train, y_train)
            self.coef_ -= self.learning_rate * gw
            self.intercept_ -= self.learning_rate * gb

            if it % 20 == 0:
                loss = cross_entropy(self.coef_, self.intercept_, X_train, y_train)
                self.loss_history_.append(loss)
                if len(self.loss_history_) > 2 and abs(self.loss_history_[-2] - self.loss_history_[-1]) < self.tol:
                    break
        return self

    def decision_function(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_

    def predict_proba(self, X):
        p1 = sigmoid(self.decision_function(X))
        return np.column_stack([1 - p1, p1])

    def predict(self, X, threshold=0.5):
        return (sigmoid(self.decision_function(X)) >= threshold).astype(int)

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


print("LogisticRegressionScratch class created successfully!")

### Step 3: Split Data and Apply Feature Scaling

Feature scaling is important for logistic regression trained with gradient descent (it accelerates convergence) and is **essential** for the regularized variants we'll see below.

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=42, stratify=y_clf
)

print(f"Training set size: {X_train_c.shape[0]}")
print(f"Test set size:     {X_test_c.shape[0]}")
print(f"Train class distribution: {np.bincount(y_train_c)}")
print(f"Test  class distribution: {np.bincount(y_test_c)}")

scaler_c = StandardScaler()
scaler_c.fit(X_train_c)
X_train_cs = scaler_c.transform(X_train_c)
X_test_cs = scaler_c.transform(X_test_c)

### Step 4: Train and evaluate the scratch logistic regression

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score

logreg_scratch = LogisticRegressionScratch(learning_rate=0.1, n_iters=3000, tol=1e-8)

start_time = time.time()
logreg_scratch.fit(X_train_cs, y_train_c)
train_time = time.time() - start_time

y_pred_scratch = logreg_scratch.predict(X_test_cs)
acc = accuracy_score(y_test_c, y_pred_scratch)

print(f"Training time: {train_time:.4f} s")
print(f"Accuracy:      {acc:.4f} ({acc * 100:.2f}%)")

plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(logreg_scratch.loss_history_)) * 20, logreg_scratch.loss_history_, "o-", markersize=3)
plt.xlabel("Iteration")
plt.ylabel("Cross-entropy loss")
plt.title("From-Scratch Logistic Regression - Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Step 5: Confusion Matrix and Classification Report

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_test_c, y_pred_scratch)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cancer.target_names)
disp.plot(cmap="Blues", ax=ax)
plt.title("Confusion Matrix - Logistic Regression (From Scratch)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(classification_report(y_test_c, y_pred_scratch, target_names=cancer.target_names))

### Step 6: Visualize Decision Boundary (2D)

In [ ]:
# Pick two informative features for a 2D visualization
feat_ix = [0, 1]  # mean radius, mean texture
X_train_2d = X_train_cs[:, feat_ix]
X_test_2d = X_test_cs[:, feat_ix]

logreg_2d = LogisticRegressionScratch(learning_rate=0.1, n_iters=3000)
logreg_2d.fit(X_train_2d, y_train_c)

h = 0.02
x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z_prob = sigmoid(logreg_2d.decision_function(np.c_[xx.ravel(), yy.ravel()])).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(12, 8))
cs = ax.contourf(xx, yy, Z_prob, levels=20, alpha=0.5, cmap="coolwarm")
ax.contour(xx, yy, Z_prob, levels=[0.5], colors="k", linewidths=2)

scatter = ax.scatter(
    X_train_2d[:, 0], X_train_2d[:, 1], c=y_train_c, cmap="coolwarm", edgecolor="black", s=60, alpha=0.8
)
plt.colorbar(cs, ax=ax, label="P(y = benign)")
ax.set_xlabel(f"standardized {cancer.feature_names[feat_ix[0]]}")
ax.set_ylabel(f"standardized {cancer.feature_names[feat_ix[1]]}")
ax.set_title("Logistic Regression - Decision Boundary (2D, From Scratch)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

acc_2d = logreg_2d.score(X_test_2d, y_test_c)
print(f"Accuracy with only 2 features: {acc_2d:.4f} ({acc_2d * 100:.2f}%)")

# 7. Logistic Regression: Scikit-learn Implementation

Scikit-learn's `LogisticRegression` is the optimized, production-ready implementation. It supports several solvers (`lbfgs`, `liblinear`, `saga`, `newton-cg`), automatic L1/L2/Elastic Net regularization, class weighting, and a consistent API with every other sklearn estimator.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Large C (=> very weak regularization) so this approaches unpenalized MLE
logreg_skl = LogisticRegression(C=1e6, solver="lbfgs", max_iter=5000)
logreg_skl.fit(X_train_cs, y_train_c)

y_pred_skl = logreg_skl.predict(X_test_cs)
y_prob_skl = logreg_skl.predict_proba(X_test_cs)[:, 1]

acc = accuracy_score(y_test_c, y_pred_skl)
prec = precision_score(y_test_c, y_pred_skl)
rec = recall_score(y_test_c, y_pred_skl)
f1 = f1_score(y_test_c, y_pred_skl)

print("=" * 60)
print("SCIKIT-LEARN LOGISTIC REGRESSION RESULTS")
print("=" * 60)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

print(classification_report(y_test_c, y_pred_skl, target_names=cancer.target_names))
print(f"Intercept: {logreg_skl.intercept_[0]:+.4f}")
print(
    f"Agreement with from-scratch coefs (max |diff|): {np.max(np.abs(logreg_skl.coef_[0] - logreg_scratch.coef_)):.4f}"
)
# Note: sklearn regularizes by default (even with huge C, numerics differ from pure MLE).
# statsmodels gives the true unpenalized MLE (see next section).

In [ ]:
# ROC curve
from sklearn.metrics import auc, roc_curve

fpr, tpr, _ = roc_curve(y_test_c, y_prob_skl)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, lw=2, label=f"ROC curve (AUC = {roc_auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve - Sklearn Logistic Regression", fontweight="bold")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 8. Logistic Regression: Statsmodels Implementation

For **inference** on logistic regression we again turn to statsmodels. The model is a **Generalized Linear Model (GLM)** with a Binomial family and a logit link. We have two equivalent APIs:

1. **`sm.Logit(y, X)`** — the direct logistic regression model.
2. **`sm.GLM(y, X, family=sm.families.Binomial())`** — the general GLM interface.

Both give the same coefficients, p-values, and confidence intervals. Remember: `sm.add_constant(X)` is required (statsmodels does not add an intercept for you).

In [ ]:
X_train_c_sm = sm.add_constant(X_train_cs)
X_test_c_sm = sm.add_constant(X_test_cs)

# Option 1: sm.Logit
logit_sm = sm.Logit(y_train_c, X_train_c_sm)
# Breast-cancer features are highly correlated, so the default Newton-Raphson
# solver hits a singular Hessian. 'bfgs' is more robust in this regime.
res_logit = logit_sm.fit(method="bfgs", maxiter=200, disp=False)

print(res_logit.summary(xname=["const"] + list(cancer.feature_names)))

In [ ]:
# Option 2: sm.GLM with a Binomial family (equivalent)
glm_sm = sm.GLM(y_train_c, X_train_c_sm, family=sm.families.Binomial())
res_glm = glm_sm.fit()
print("sm.Logit vs sm.GLM coefficient agreement:")
print(f"  Max |diff| = {np.max(np.abs(res_logit.params - res_glm.params)):.2e}")

# Build a clean coefficient table with odds ratios
coef_table = pd.DataFrame(
    {
        "coef (log-odds)": res_logit.params,
        "std_err": res_logit.bse,
        "z": res_logit.tvalues,
        "p>|z|": res_logit.pvalues,
        "CI_low": res_logit.conf_int()[:, 0],
        "CI_high": res_logit.conf_int()[:, 1],
        "odds_ratio": np.exp(res_logit.params),
    },
    index=["const"] + list(cancer.feature_names),
)

print("\nTop 10 significant features (by |z|):")
coef_table_abs = coef_table.assign(abs_z=coef_table["z"].abs()).sort_values("abs_z", ascending=False).head(10)
print(coef_table_abs.round(4))

# Pseudo-R^2 (McFadden's) and model statistics
print(f"\nLog-likelihood:          {res_logit.llf:.2f}")
print(f"LL-Null (intercept-only): {res_logit.llnull:.2f}")
print(f"Pseudo R^2 (McFadden):    {res_logit.prsquared:.4f}")
print(f"LLR p-value:              {res_logit.llr_pvalue:.2e}")
print(f"AIC:                      {res_logit.aic:.2f}")
print(f"BIC:                      {res_logit.bic:.2f}")

### Interpreting logistic regression coefficients

In logistic regression each coefficient $w_j$ is on the **log-odds** scale:

* **coef = 0** → feature has no effect on the odds.
* **coef > 0** → feature increases the probability of class 1.
* **odds_ratio = $e^{w_j}$** → a one-unit (here: 1-std, since we standardized) increase in the feature multiplies the odds of class 1 by $e^{w_j}$.

For example, `odds_ratio = 2.0` means: “a one-unit increase in this feature doubles the odds of class 1, holding everything else fixed”.

**Model-level diagnostics:**
* **Pseudo R² (McFadden)**: $1 - \tfrac{\text{LL}_\text{full}}{\text{LL}_\text{null}}$. Values between 0.2 and 0.4 indicate excellent fit. Not comparable to linear-regression R².
* **LLR p-value**: Likelihood-ratio test for "all coefficients are zero".
* **AIC / BIC**: Information criteria for model comparison.

### Prediction with statsmodels

`res_logit.predict(X)` returns **probabilities** (not hard labels). Threshold at 0.5 (or a different value based on your cost function) to get class predictions.

In [ ]:
# Prediction using statsmodels
y_prob_sm = res_logit.predict(X_test_c_sm)
y_pred_sm_bin = (y_prob_sm >= 0.5).astype(int)

print(f"statsmodels accuracy: {accuracy_score(y_test_c, y_pred_sm_bin):.4f}")
print(f"Agreement with sklearn predictions: {np.mean(y_pred_sm_bin == y_pred_skl):.4f}")

## 8.1 The Formula API (R-style)

Just like `smf.ols`, statsmodels exposes `smf.logit` / `smf.glm` with the formula interface. This is particularly handy when you have categorical variables — statsmodels will dummy-encode them automatically.

In [ ]:
# Using only a handful of features for a compact demo
feature_subset = list(cancer.feature_names[:5])

train_df_c = pd.DataFrame(X_train_cs, columns=cancer.feature_names)
train_df_c["target"] = y_train_c

# Remember to sanitize feature names with spaces for the formula interface.
safe_names = [f.replace(" ", "_") for f in feature_subset]
train_df_c_renamed = train_df_c.rename(columns=dict(zip(feature_subset, safe_names)))

formula = "target ~ " + " + ".join(safe_names)
print(f"Formula: {formula}\n")

logit_formula = smf.logit(formula=formula, data=train_df_c_renamed).fit(disp=False)
print(logit_formula.summary().tables[1])

# 9. Regularized Logistic Regression

Just like linear regression, logistic regression can be regularized with L1, L2, or Elastic Net penalties. The loss becomes:

$$\mathcal{L}(w,b) = \text{CrossEntropy}(w,b) + \lambda\,\Omega(w)$$

where $\Omega(w)$ is the penalty (L1, L2, or a mix).

**Sklearn convention**: regularization strength is `C = 1 / \lambda`. Large C = weak regularization; small C = heavy regularization.

Supported solvers for each penalty:

| `penalty`    | Compatible `solver`                       |
|--------------|-------------------------------------------|
| `None`       | `lbfgs`, `newton-cg`, `sag`, `saga`       |
| `l2`         | all solvers (default)                     |
| `l1`         | `liblinear`, `saga`                       |
| `elasticnet` | `saga` only                               |

## 9.1 L2-Regularized Logistic Regression (default)

In [ ]:
l2_logreg = LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000)
l2_logreg.fit(X_train_cs, y_train_c)
y_pred_l2 = l2_logreg.predict(X_test_cs)

print("=" * 60)
print("L2-PENALIZED LOGISTIC REGRESSION (C=1.0)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_l2):.4f}")
print(f"F1:       {f1_score(y_test_c, y_pred_l2):.4f}")
print(f"Non-zero coefficients: {(l2_logreg.coef_[0] != 0).sum()} / {X_train_cs.shape[1]}")

## 9.2 L1-Regularized Logistic Regression (feature selection)

In [ ]:
# liblinear supports L1 and is fast for small/medium binary problems
l1_logreg = LogisticRegression(C=0.1, l1_ratio=1.0, solver="saga", max_iter=5000)
l1_logreg.fit(X_train_cs, y_train_c)
y_pred_l1 = l1_logreg.predict(X_test_cs)

print("=" * 60)
print("L1-PENALIZED LOGISTIC REGRESSION (C=0.1)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_l1):.4f}")
print(f"F1:       {f1_score(y_test_c, y_pred_l1):.4f}")
n_nonzero_l1 = (l1_logreg.coef_[0] != 0).sum()
print(
    f"Non-zero coefficients: {n_nonzero_l1} / {X_train_cs.shape[1]} "
    f"(= {n_nonzero_l1 / X_train_cs.shape[1] * 100:.0f}% selected)"
)

l1_df = pd.DataFrame({"feature": cancer.feature_names, "coef": l1_logreg.coef_[0]})
l1_df = l1_df[l1_df["coef"] != 0].copy()
l1_df["abs_coef"] = l1_df["coef"].abs()
l1_df = l1_df.sort_values("abs_coef", ascending=False).head(10)
print("\nTop selected features:")
print(l1_df[["feature", "coef"]].to_string(index=False))

## 9.3 Elastic Net Logistic Regression

In [ ]:
# saga solver supports elasticnet
enet_logreg = LogisticRegression(C=0.5, l1_ratio=0.5, solver="saga", max_iter=10000)
enet_logreg.fit(X_train_cs, y_train_c)
y_pred_enet_l = enet_logreg.predict(X_test_cs)

print("=" * 60)
print("ELASTIC NET LOGISTIC REGRESSION (C=0.5, l1_ratio=0.5)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_enet_l):.4f}")
print(f"F1:       {f1_score(y_test_c, y_pred_enet_l):.4f}")
print(f"Non-zero coefficients: {(enet_logreg.coef_[0] != 0).sum()} / {X_train_cs.shape[1]}")

## 9.4 Coefficient Paths: C Sweep

In [ ]:
Cs = np.logspace(-3, 3, 30)
paths_l2, paths_l1 = [], []
acc_l2, acc_l1 = [], []
for C in Cs:
    m_l2 = LogisticRegression(C=C, solver="lbfgs", max_iter=5000).fit(X_train_cs, y_train_c)
    m_l1 = LogisticRegression(C=C, l1_ratio=1.0, solver="saga", max_iter=5000).fit(X_train_cs, y_train_c)
    paths_l2.append(m_l2.coef_[0])
    paths_l1.append(m_l1.coef_[0])
    acc_l2.append(m_l2.score(X_test_cs, y_test_c))
    acc_l1.append(m_l1.score(X_test_cs, y_test_c))
paths_l2 = np.array(paths_l2)
paths_l1 = np.array(paths_l1)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for j in range(paths_l2.shape[1]):
    axes[0, 0].semilogx(Cs, paths_l2[:, j], alpha=0.5, lw=1)
axes[0, 0].axhline(0, color="black", lw=1)
axes[0, 0].set_title("L2 coefficient path", fontweight="bold")
axes[0, 0].set_xlabel("C (log scale)")
axes[0, 0].set_ylabel("Coefficient")
axes[0, 0].grid(True, alpha=0.3)

for j in range(paths_l1.shape[1]):
    axes[0, 1].semilogx(Cs, paths_l1[:, j], alpha=0.5, lw=1)
axes[0, 1].axhline(0, color="black", lw=1)
axes[0, 1].set_title("L1 coefficient path", fontweight="bold")
axes[0, 1].set_xlabel("C (log scale)")
axes[0, 1].set_ylabel("Coefficient")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].semilogx(Cs, acc_l2, "o-", label="L2", markersize=4)
axes[1, 0].semilogx(Cs, acc_l1, "s-", label="L1", markersize=4)
axes[1, 0].set_title("Test Accuracy vs C", fontweight="bold")
axes[1, 0].set_xlabel("C (log scale)")
axes[1, 0].set_ylabel("Accuracy")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

sparsity_l1 = (paths_l1 != 0).sum(axis=1)
axes[1, 1].semilogx(Cs, sparsity_l1, "o-", markersize=4, color="tab:green")
axes[1, 1].set_title("L1 sparsity (# non-zero coefficients) vs C", fontweight="bold")
axes[1, 1].set_xlabel("C (log scale)")
axes[1, 1].set_ylabel("# non-zero")
axes[1, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best C for L2 by test accuracy: {Cs[int(np.argmax(acc_l2))]:.4f}  (acc = {max(acc_l2):.4f})")
print(f"Best C for L1 by test accuracy: {Cs[int(np.argmax(acc_l1))]:.4f}  (acc = {max(acc_l1):.4f})")

## 9.5 Regularized Logistic Regression via Statsmodels

Statsmodels' `Logit` model also supports L1 / Elastic Net regularization through `fit_regularized`. The parameterization matches the OLS version:
* `alpha` — same meaning as in sklearn's regularization.
* `L1_wt` — analogous to sklearn's `l1_ratio`.

Again: regularized fits in statsmodels do **not** return p-values / standard errors.

In [ ]:
# Pure L1 (Lasso) logistic regression in statsmodels
logit_l1_sm = sm.Logit(y_train_c, X_train_c_sm).fit_regularized(alpha=0.1, L1_wt=1.0, disp=False)
# Pure L2 (Ridge) logistic regression in statsmodels
logit_l2_sm = sm.Logit(y_train_c, X_train_c_sm).fit_regularized(alpha=0.1, L1_wt=0.0, disp=False)
# Elastic Net
logit_enet_sm = sm.Logit(y_train_c, X_train_c_sm).fit_regularized(alpha=0.1, L1_wt=0.5, disp=False)

# Compare predictive performance


def eval_sm(fit, X_test_full):
    prob = fit.predict(X_test_full)
    pred = (prob >= 0.5).astype(int)
    return accuracy_score(y_test_c, pred), f1_score(y_test_c, pred), (fit.params[1:] != 0).sum()


for name, fit in [("Ridge (sm)", logit_l2_sm), ("Lasso (sm)", logit_l1_sm), ("Elastic Net (sm)", logit_enet_sm)]:
    acc_sm, f1_sm, nz = eval_sm(fit, X_test_c_sm)
    print(f"{name:<18}  acc={acc_sm:.4f}  F1={f1_sm:.4f}  non-zero coefs={nz}/{X_train_cs.shape[1]}")

## 9.6 Side-by-Side Comparison: Unpenalized vs L1 vs L2 vs Elastic Net

In [ ]:
models_logit = {
    "Unpenalized (C=1e6, lbfgs)": LogisticRegression(C=1e6, solver="lbfgs", max_iter=5000),
    "L2 (C=1, lbfgs)": LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000),
    "L1 (C=0.1, liblinear)": LogisticRegression(C=0.1, l1_ratio=1.0, solver="saga", max_iter=5000),
    "Elastic Net (C=0.5, rho=0.5, saga)": LogisticRegression(C=0.5, l1_ratio=0.5, solver="saga", max_iter=10000),
}

rows_l = []
coef_matrix_l = {}
for name, m in models_logit.items():
    m.fit(X_train_cs, y_train_c)
    y_pred = m.predict(X_test_cs)
    y_prob = m.predict_proba(X_test_cs)[:, 1]
    rows_l.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test_c, y_pred),
            "precision": precision_score(y_test_c, y_pred),
            "recall": recall_score(y_test_c, y_pred),
            "f1": f1_score(y_test_c, y_pred),
            "auc": auc(*roc_curve(y_test_c, y_prob)[:2]),
            "n_nonzero": int((m.coef_[0] != 0).sum()),
        }
    )
    coef_matrix_l[name] = m.coef_[0]

print("=" * 95)
print("LOGISTIC REGRESSION VARIANT COMPARISON")
print("=" * 95)
print(pd.DataFrame(rows_l).round(4).to_string(index=False))

In [ ]:
# Visualize coefficients side-by-side
fig, axes = plt.subplots(1, 4, figsize=(20, 8), sharey=True)
for ax, (name, vec) in zip(axes, coef_matrix_l.items()):
    colors = ["tab:blue" if v > 0 else "tab:red" for v in vec]
    ax.barh(range(len(vec)), vec, color=colors, edgecolor="black", alpha=0.8)
    ax.axvline(0, color="black", lw=1)
    ax.set_title(name, fontweight="bold", fontsize=10)
    ax.grid(True, alpha=0.3, axis="x")
axes[0].set_ylabel("Feature index")
plt.suptitle("Logistic Regression Coefficients by Penalty", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# 10. Cross-Validation for Model Evaluation

Cross-validation is a crucial technique for assessing how well a model generalizes to unseen data. It helps us avoid overfitting and provides more reliable performance estimates than a single train-test split.

## Why Cross-Validation?

**Problem with single train-test split:**
- Results can vary significantly depending on which samples end up in training vs. test
- May get lucky (or unlucky) with the split
- Doesn't fully utilize the available data for training

**Solution: Cross-Validation**
- Systematically rotate through different train-test combinations
- More robust performance estimate
- Better use of limited data
- Helps detect overfitting / underfitting

## 10.1 Hold-out Approach (Baseline)

In [ ]:
print("=" * 60)
print("HOLD-OUT APPROACH")
print("=" * 60)

# Regression hold-out
lr_ho = LinearRegression().fit(X_train_rs, y_train_r)
print(f"[Regression]     Test R^2: {lr_ho.score(X_test_rs, y_test_r):.4f}")

# Classification hold-out
lg_ho = LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000).fit(X_train_cs, y_train_c)
print(f"[Classification] Test accuracy: {lg_ho.score(X_test_cs, y_test_c):.4f}")

print("\nLIMITATIONS OF HOLD-OUT")
print("1. Performance estimate depends on the specific split")
print("2. Some data is never used for training")
print("3. Can give overly optimistic or pessimistic estimates")
print("4. Not ideal for small datasets")

## 10.2 K-Fold Cross-Validation (Manual)

**How K-Fold Cross-Validation Works:**
1. Split the dataset into K equal-sized folds.
2. For each of the K iterations:
   - Use 1 fold as test, the remaining K-1 folds as train.
   - Train and evaluate the model.
3. Average the K performance scores.

For classification, use **StratifiedKFold** to keep class balance stable across folds.

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold

print("=" * 60)
print("K-FOLD CV (MANUAL) - REGRESSION (Ridge)")
print("=" * 60)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores_reg = []
for i, (tr, te) in enumerate(kf.split(X_reg), 1):
    X_tr, X_te = X_reg[tr], X_reg[te]
    y_tr, y_te = y_reg[tr], y_reg[te]
    sc = StandardScaler()
    m = Ridge(alpha=1.0).fit(sc.fit_transform(X_tr), y_tr)
    s = m.score(sc.transform(X_te), y_te)
    fold_scores_reg.append(s)
    print(f"Fold {i}: R^2 = {s:.4f}")
print(f"\nMean R^2: {np.mean(fold_scores_reg):.4f} (+/- {np.std(fold_scores_reg):.4f})")

print("\n" + "=" * 60)
print("STRATIFIED K-FOLD CV (MANUAL) - CLASSIFICATION (L2 logistic)")
print("=" * 60)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores_clf = []
for i, (tr, te) in enumerate(skf.split(X_clf, y_clf), 1):
    X_tr, X_te = X_clf[tr], X_clf[te]
    y_tr, y_te = y_clf[tr], y_clf[te]
    sc = StandardScaler()
    m = LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000).fit(sc.fit_transform(X_tr), y_tr)
    s = m.score(sc.transform(X_te), y_te)
    fold_scores_clf.append(s)
    print(f"Fold {i}: accuracy = {s:.4f}")
print(f"\nMean accuracy: {np.mean(fold_scores_clf):.4f} (+/- {np.std(fold_scores_clf):.4f})")

## 10.3 Using `cross_validate()`

Sklearn provides `cross_validate()` which automates cross-validation and can compute multiple metrics simultaneously. It also returns both training and test scores.

Pipelines ensure preprocessing (scaling) is fit only on the training portion of each fold — preventing data leakage.

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

# Regression pipeline
pipe_reg = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
cv_reg = cross_validate(
    pipe_reg,
    X_reg,
    y_reg,
    cv=5,
    scoring={"r2": "r2", "rmse": "neg_root_mean_squared_error"},
    return_train_score=True,
    n_jobs=-1,
)
print("REGRESSION CROSS_VALIDATE() RESULTS")
print("-" * 40)
print(f"Test R^2:  {np.mean(cv_reg['test_r2']):.4f} (+/- {np.std(cv_reg['test_r2']):.4f})")
print(f"Test RMSE: {-np.mean(cv_reg['test_rmse']):.4f} (+/- {np.std(cv_reg['test_rmse']):.4f})")

# Classification pipeline
pipe_clf = Pipeline(
    [("scaler", StandardScaler()), ("logreg", LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000))]
)
cv_clf = cross_validate(
    pipe_clf,
    X_clf,
    y_clf,
    cv=5,
    scoring={"accuracy": "accuracy", "f1": "f1", "auc": "roc_auc"},
    return_train_score=True,
    n_jobs=-1,
)
print("\nCLASSIFICATION CROSS_VALIDATE() RESULTS")
print("-" * 40)
print(f"Test accuracy: {np.mean(cv_clf['test_accuracy']):.4f} (+/- {np.std(cv_clf['test_accuracy']):.4f})")
print(f"Test F1:       {np.mean(cv_clf['test_f1']):.4f} (+/- {np.std(cv_clf['test_f1']):.4f})")
print(f"Test AUC:      {np.mean(cv_clf['test_auc']):.4f} (+/- {np.std(cv_clf['test_auc']):.4f})")

## 10.4 Built-in CV Variants: `RidgeCV`, `LassoCV`, `ElasticNetCV`, `LogisticRegressionCV`

Sklearn offers CV-aware versions of the regularized estimators that efficiently scan the `alpha` (or `C`) grid internally. They are typically *much* faster than wrapping a plain estimator in `GridSearchCV`, because they reuse computations across the regularization path.

In [ ]:
from sklearn.linear_model import ElasticNetCV, LassoCV, LogisticRegressionCV, RidgeCV

# Regression: tune alpha internally
ridge_cv = RidgeCV(alphas=np.logspace(-4, 4, 50), cv=5).fit(X_train_rs, y_train_r)
print(f"RidgeCV:      best alpha = {ridge_cv.alpha_:.4f}   test R^2 = {ridge_cv.score(X_test_rs, y_test_r):.4f}")

lasso_cv = LassoCV(n_alphas=100, cv=5, max_iter=20000, random_state=42).fit(X_train_rs, y_train_r)
print(f"LassoCV:      best alpha = {lasso_cv.alpha_:.6f}   test R^2 = {lasso_cv.score(X_test_rs, y_test_r):.4f}")

enet_cv = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0], n_alphas=50, cv=5, max_iter=20000, random_state=42).fit(
    X_train_rs, y_train_r
)
print(
    f"ElasticNetCV: best alpha = {enet_cv.alpha_:.6f}   best l1_ratio = {enet_cv.l1_ratio_:.2f}   "
    f"test R^2 = {enet_cv.score(X_test_rs, y_test_r):.4f}"
)

# Classification: tune C internally
logreg_cv = LogisticRegressionCV(
    Cs=np.logspace(-3, 3, 20),
    solver="lbfgs",
    cv=5,
    max_iter=5000,
    n_jobs=-1,
).fit(X_train_cs, y_train_c)
print(
    f"\nLogisticRegressionCV (L2): best C = {logreg_cv.C_[0]:.4f}   "
    f"test accuracy = {logreg_cv.score(X_test_cs, y_test_c):.4f}"
)

# 11. Hyperparameter Tuning with Cross-Validation

**Hyperparameters** are model settings that we choose before training (unlike parameters which are learned during training).

For regularized linear / logistic regression, the key hyperparameters are:
- **`alpha`** (linear) or **`C = 1/alpha`** (logistic): regularization strength (always log-scale).
- **`l1_ratio`** (Elastic Net): L1/L2 mixing in $[0, 1]$.
- **`penalty`** (logistic): `'l1'`, `'l2'`, `'elasticnet'`, or `None`.
- **`solver`**: must be compatible with the chosen penalty.

**Goal**: find the hyperparameter combination that gives the best cross-validation performance.

## 11.1 Manual Grid Search

In [ ]:
from sklearn.model_selection import cross_val_score

print("=" * 60)
print("MANUAL GRID SEARCH - Elastic Net Logistic Regression")
print("=" * 60)

C_grid = np.logspace(-2, 2, 6)
l1_grid = [0.1, 0.5, 0.9]

records = []
total = len(C_grid) * len(l1_grid)
k = 0
for C in C_grid:
    for rho in l1_grid:
        k += 1
        pipe = Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "logreg",
                    LogisticRegression(
                        C=C,
                        l1_ratio=rho,
                        solver="saga",
                        max_iter=10000,
                    ),
                ),
            ]
        )
        scores = cross_val_score(pipe, X_clf, y_clf, cv=5, scoring="accuracy", n_jobs=-1)
        mean_s, std_s = scores.mean(), scores.std()
        records.append({"C": C, "l1_ratio": rho, "mean_acc": mean_s, "std_acc": std_s})
        print(f"[{k:2d}/{total}] C={C:<8.3f} l1_ratio={rho:<5}  acc={mean_s:.4f} (+/- {std_s:.4f})")

manual_df = pd.DataFrame(records).sort_values("mean_acc", ascending=False)
print("\nTOP 5 CONFIGURATIONS")
print(manual_df.head().to_string(index=False))

## 11.2 GridSearchCV (Automated Grid Search)

Sklearn's `GridSearchCV` automates the grid search process with additional features:
- Parallel processing
- Automatic cross-validation
- Detailed results tracking
- Easy access to best parameters and scores

In [ ]:
from sklearn.model_selection import GridSearchCV

# Grid search for the regularized linear regression (ElasticNet on housing data)
pipe_gs_reg = Pipeline([("scaler", StandardScaler()), ("enet", ElasticNet(max_iter=20000))])
grid_reg = GridSearchCV(
    pipe_gs_reg,
    param_grid={
        "enet__alpha": np.logspace(-4, 1, 8),
        "enet__l1_ratio": [0.1, 0.5, 0.9, 1.0],
    },
    cv=5,
    scoring="r2",
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
)
grid_reg.fit(X_reg, y_reg)
print("[REGRESSION] Best params:", grid_reg.best_params_)
print(f"[REGRESSION] Best CV R^2: {grid_reg.best_score_:.4f}")

# Grid search for L2 logistic regression (classification)
pipe_gs_clf = Pipeline([("scaler", StandardScaler()), ("logreg", LogisticRegression(solver="lbfgs", max_iter=5000))])
grid_clf = GridSearchCV(
    pipe_gs_clf,
    param_grid={
        "logreg__C": np.logspace(-3, 3, 10),
    },
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
)
grid_clf.fit(X_clf, y_clf)
print("\n[CLASSIFICATION] Best params:", grid_clf.best_params_)
print(f"[CLASSIFICATION] Best CV accuracy: {grid_clf.best_score_:.4f}")

## 11.3 RandomizedSearchCV (Automated Random Search)

`RandomizedSearchCV` is particularly useful when:
- The hyperparameter space is very large
- You have limited computational budget
- You want to explore continuous distributions

For `alpha` / `C`, the natural distribution is **log-uniform**: we want equal coverage of each order of magnitude.

In [ ]:
from scipy.stats import loguniform, uniform
from sklearn.model_selection import RandomizedSearchCV

# Elastic Net logistic regression, random search
pipe_rs = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(solver="saga", max_iter=10000)),
    ]
)

random_search = RandomizedSearchCV(
    estimator=pipe_rs,
    param_distributions={
        "logreg__C": loguniform(1e-3, 1e3),
        "logreg__l1_ratio": uniform(loc=0.0, scale=1.0),  # uniform on [0, 1]
    },
    n_iter=30,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    return_train_score=True,
    verbose=1,
)
random_search.fit(X_clf, y_clf)

print("=" * 60)
print("RANDOMIZEDSEARCHCV RESULTS (Elastic Net logistic regression)")
print("=" * 60)
print("Best params:", random_search.best_params_)
print(f"Best CV accuracy: {random_search.best_score_:.4f}")

print("\n" + "=" * 60)
print("COMPARISON: GRIDSEARCH vs RANDOMIZEDSEARCH")
print("=" * 60)
print(
    f"GridSearchCV (L2 logreg):         best accuracy = {grid_clf.best_score_:.4f}  "
    f"({len(grid_clf.cv_results_['params'])} combos)"
)
print(f"RandomizedSearchCV (enet logreg): best accuracy = {random_search.best_score_:.4f}  (30 combos)")

# 12. Comprehensive Examples: Complete Pipelines

Let's put everything together into complete, production-ready pipelines that incorporate all best practices we've learned.

## Complete Workflow:
1. Load and explore data
2. Split into train/test sets
3. Build pipeline with preprocessing
4. Hyperparameter tuning with cross-validation
5. Evaluate final model on test set
6. Analyze results

## 12.1 Complete Regression Pipeline (Elastic Net)

In [ ]:
print("=" * 70)
print("COMPLETE REGRESSION PIPELINE (Elastic Net on Housing)")
print("=" * 70)

# Step 1: load data (already loaded)
print(f"\n[STEP 1] Dataset shape: {X_reg.shape}")

# Step 2: train/test split BEFORE preprocessing
X_tr_final, X_te_final, y_tr_final, y_te_final = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
print(f"\n[STEP 2] Training: {X_tr_final.shape[0]}   Test: {X_te_final.shape[0]}")

# Step 3: pipeline
pipeline_final_reg = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("enet", ElasticNet(max_iter=20000, random_state=42)),
    ]
)
print("\n[STEP 3] Pipeline: StandardScaler -> ElasticNet")

# Step 4: hyperparameter tuning
grid_reg_final = GridSearchCV(
    pipeline_final_reg,
    param_grid={
        "enet__alpha": np.logspace(-4, 1, 10),
        "enet__l1_ratio": [0.0, 0.3, 0.5, 0.7, 1.0],
    },
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid_reg_final.fit(X_tr_final, y_tr_final)
print("\n[STEP 4] Best params:", grid_reg_final.best_params_)
print(f"         Best CV R^2: {grid_reg_final.best_score_:.4f}")

# Step 5: evaluate on held-out test set
best_reg = grid_reg_final.best_estimator_
y_pred_reg = best_reg.predict(X_te_final)
print("\n[STEP 5] FINAL TEST METRICS")
print(f"  R^2:  {r2_score(y_te_final, y_pred_reg):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_te_final, y_pred_reg)):.4f}")
print(f"  MAE:  {mean_absolute_error(y_te_final, y_pred_reg):.4f}")

## 12.2 Complete Classification Pipeline (Logistic Regression)

In [ ]:
print("=" * 70)
print("COMPLETE CLASSIFICATION PIPELINE (Logistic Regression on Breast Cancer)")
print("=" * 70)

# Step 1
print(f"\n[STEP 1] Dataset shape: {X_clf.shape}   Classes: {cancer.target_names}")

# Step 2: stratified split
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
print(f"\n[STEP 2] Training: {X_tr_c.shape[0]}   Test: {X_te_c.shape[0]}")

# Step 3: pipeline
pipeline_final_clf = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(solver="saga", max_iter=10000, random_state=42)),
    ]
)
print("\n[STEP 3] Pipeline: StandardScaler -> LogisticRegression(saga)")

# Step 4: hyperparameter tuning (both penalty, C, and l1_ratio)
grid_clf_final = GridSearchCV(
    pipeline_final_clf,
    param_grid={
        "logreg__C": np.logspace(-3, 3, 10),
        "logreg__l1_ratio": [0.0, 0.3, 0.5, 0.7, 1.0],  # 0=L2, 1=L1, in between=Elastic Net
    },
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
grid_clf_final.fit(X_tr_c, y_tr_c)
print("\n[STEP 4] Best params:", grid_clf_final.best_params_)
print(f"         Best CV accuracy: {grid_clf_final.best_score_:.4f}")

# Step 5: evaluate on held-out test set
best_clf = grid_clf_final.best_estimator_
y_pred_clf = best_clf.predict(X_te_c)
y_prob_clf = best_clf.predict_proba(X_te_c)[:, 1]

print("\n[STEP 5] FINAL TEST METRICS")
print(f"  Accuracy:  {accuracy_score(y_te_c, y_pred_clf):.4f}")
print(f"  Precision: {precision_score(y_te_c, y_pred_clf):.4f}")
print(f"  Recall:    {recall_score(y_te_c, y_pred_clf):.4f}")
print(f"  F1-score:  {f1_score(y_te_c, y_pred_clf):.4f}")
print(f"  AUC:       {auc(*roc_curve(y_te_c, y_prob_clf)[:2]):.4f}")

print("\nDETAILED CLASSIFICATION REPORT")
print(classification_report(y_te_c, y_pred_clf, target_names=cancer.target_names))

In [ ]:
# Final confusion matrix
cm_final = confusion_matrix(y_te_c, y_pred_clf)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_final, display_labels=cancer.target_names)
disp.plot(cmap="Greens", ax=ax, values_format="d")
plt.title("Final Logistic Regression Model - Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# 13. Key Takeaways and Best Practices

## Summary

**Strengths:**
- ✅ Simple, fast, and highly interpretable.
- ✅ Convex optimization — closed-form (OLS/Ridge) or unique global minimum (logistic).
- ✅ Natural probabilistic interpretation (MLE under Gaussian or Bernoulli noise).
- ✅ Statistical inference is available through `statsmodels` (p-values, CIs, F-tests).
- ✅ Regularized variants handle multicollinearity and high-dimensional problems.
- ✅ Lasso / L1 gives automatic variable selection.
- ✅ Strong baseline for any regression or classification task.

**Weaknesses:**
- ❌ Linearity (or log-linearity) assumption — misses non-linear relationships unless you feature-engineer them.
- ❌ Sensitive to outliers (OLS squared loss; logistic is milder but still affected).
- ❌ Unregularized variants blow up under multicollinearity.
- ❌ Require feature scaling for regularized variants.
- ❌ Classical inference relies on strong assumptions.

## Best Practices

### 1. **ALWAYS Scale for Regularized Models**
```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0)),         # or LogisticRegression(C=1.0)
])
```

### 2. **Use Pipelines and Cross-Validation Together**
- Prevents data leakage (scaling fit on training folds only).
- Everything works seamlessly with `GridSearchCV` / `cross_validate`.

### 3. **Search `alpha` / `C` on a Log Scale**
- `alpha ∈ {1e-4, ..., 1e4}` or `C ∈ {1e-4, ..., 1e4}` — a linear grid is almost always wrong.
- For `l1_ratio`, a short uniform grid `[0.1, 0.3, 0.5, 0.7, 0.9, 1.0]` is usually enough.

### 4. **Pick the Right Regularization**
- **Plain (OLS / unpenalized logistic)**: when $n \gg p$ and features are uncorrelated.
- **Ridge (L2)**: when features are correlated but all are relevant.
- **Lasso (L1)**: when you believe the true model is sparse (few features matter).
- **Elastic Net**: when features are correlated *and* you want sparsity.

### 5. **Use Built-in CV Variants for Speed**
- `RidgeCV`, `LassoCV`, `ElasticNetCV`, `LogisticRegressionCV` exploit the solution path — orders of magnitude faster than `GridSearchCV`.

### 6. **Match Solver to Penalty (Logistic)**
| `penalty` | Compatible `solver` |
|---|---|
| `None`, `l2` | `lbfgs` (default), `newton-cg`, `sag`, `saga`, `liblinear` |
| `l1` | `liblinear`, `saga` |
| `elasticnet` | `saga` only |

### 7. **Use `statsmodels` for Inference**
- For OLS: `sm.OLS`, for logistic: `sm.Logit` or `sm.GLM(family=Binomial())`.
- Remember to `sm.add_constant(X)`!
- The formula API (`smf.ols`, `smf.logit`, `smf.glm`) handles categorical variables automatically.
- For regularized fits use `fit_regularized()` — but note the p-values / standard errors are NOT returned (the sampling distribution of penalized estimators is non-standard).

### 8. **Always Look at Residual / Diagnostic Plots**
- **Regression**: residuals vs fitted, Q-Q plot, histogram of residuals.
- **Classification**: ROC curve, precision-recall curve, confusion matrix, calibration plot.
- If you see a pattern in residuals or poor calibration, the model is misspecified.

### 9. **Never Leak Test Data**
- Scale and impute inside a `Pipeline` so CV folds are isolated.
- Tune hyperparameters on CV, then evaluate once on the test set.

### 10. **Choose the Right Metrics**
- **Regression**: RMSE (same units as target), MAE (robust), R² (unitless).
- **Classification**: accuracy is misleading on imbalanced data — prefer F1, AUC, precision-recall.
- Always report confidence intervals from CV (mean ± std).

## When to Use Linear / Logistic Regression

✅ **Use these models when:**
- You need a fast, interpretable baseline.
- You want statistical inference (hypothesis tests on coefficients).
- The relationship is approximately linear (or the log-odds are).
- You have more samples than features ($n \gg p$), or you use regularization.
- Training speed and prediction latency matter.

❌ **Avoid these models when:**
- The true relationship is strongly non-linear (consider trees, GBMs, neural nets).
- Residuals show strong heteroskedasticity or autocorrelation (consider GLS, robust regression, time-series models).
- You have heavy-tailed outliers (consider Huber, quantile regression).
- The decision boundary is clearly curved and feature engineering is not an option.

## Common Pitfalls to Avoid

1. **Forgetting to scale features before Ridge/Lasso/Elastic Net / regularized logistic** → penalty becomes unfair.
2. **Searching `alpha` / `C` on a linear grid** → wastes compute.
3. **Fitting the scaler on the full dataset** → data leakage.
4. **Mismatched `penalty` / `solver` combination** in sklearn's `LogisticRegression` → silent fallback or error.
5. **Interpreting Lasso coefficients as "causal"** → feature selection ≠ causal discovery.
6. **Reading p-values from regularized fits** → their sampling distributions are non-standard.
7. **Forgetting to `sm.add_constant(X)` in statsmodels** → silently fits without an intercept!
8. **Using accuracy as the only metric on imbalanced data** → model might “predict the majority class” and look great.
9. **Ignoring multicollinearity in OLS / unpenalized logistic** → coefficients become unstable; use VIF diagnostics or switch to Ridge.
10. **Not checking residuals / calibration** → high R² or accuracy can still hide systematic model misspecification.

## Further Reading

- [Scikit-learn Linear Models](https://scikit-learn.org/stable/modules/linear_model.html)
- [Scikit-learn Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Statsmodels Regression Overview](https://www.statsmodels.org/stable/regression.html)
- [Statsmodels GLM (incl. logistic)](https://www.statsmodels.org/stable/glm.html)
- [Elements of Statistical Learning — Chapters 3–4](https://hastie.su.domains/ElemStatLearn/)
- [Zou & Hastie (2005) — Regularization and variable selection via the elastic net](https://web.stanford.edu/~hastie/Papers/B67.2%20(2005)%20301-320%20Zou%20&%20Hastie.pdf)
- [Cross-Validation Techniques](https://scikit-learn.org/stable/modules/cross_validation.html)


# 14. Homework Assignment

You will build and compare regularized linear and logistic regression models on two datasets.

---

## Part A — Regression: Diabetes Progression

### Dataset
Use the **Diabetes dataset** from sklearn (`sklearn.datasets.load_diabetes`):
- 442 patients, 10 baseline features (age, BMI, blood pressure, 6 blood serum measurements, ...)
- Target: a quantitative measure of disease progression one year after baseline
- Features are already centered and scaled to unit norm by sklearn.

### Tasks

#### A1: Data Exploration and Preprocessing
1. Load the Diabetes dataset.
2. Explore: shape, feature names, target statistics, correlation matrix heatmap, feature-vs-target scatterplots.
3. Split into train (70%) / test (30%) with `random_state=42`.

#### A2: From-Scratch and Sklearn
1. Implement closed-form OLS with `np.linalg.solve` (Section 2.1).
2. Fit `sklearn.linear_model.LinearRegression` and verify the coefficients match.
3. Report R², RMSE, MAE on the held-out test set.

#### A3: Statsmodels Inference
1. Fit `statsmodels.api.OLS` (remember to `sm.add_constant(X)`!).
2. Interpret the summary:
   - Which features are statistically significant at $\alpha=0.05$?
   - What is the adjusted R²?
   - Comment on the condition number and the Jarque-Bera p-value.

#### A4: Regularization
1. Fit `Ridge`, `Lasso`, `ElasticNet`.
2. Plot the **coefficient path** for each as a function of `alpha` on a log scale ($10^{-3}$ to $10^{1}$).
3. Compare which features Lasso keeps vs. which Ridge keeps.
4. Replicate with `statsmodels.OLS.fit_regularized` and compare coefficients.

#### A5: Cross-Validation and Hyperparameter Tuning
1. Manual 5-fold CV with `KFold` (Ridge). Report mean ± std R².
2. Use `cross_validate()` with multiple metrics.
3. Use `RidgeCV`, `LassoCV`, `ElasticNetCV` for automatic alpha selection.
4. Use `GridSearchCV` with:
   - `alpha`: `np.logspace(-4, 1, 10)`
   - `l1_ratio`: `[0.1, 0.3, 0.5, 0.7, 0.9]`
5. Use `RandomizedSearchCV` with `loguniform(1e-4, 1e1)` / `uniform(0,1)`, at least 40 iterations.
6. Compare results, runtimes, and visualize the search (heatmap for grid, scatter for random).

#### A6: Final Model and Analysis
1. Refit the best model on the full training set.
2. Evaluate on the held-out test set (R², RMSE, MAE).
3. Produce residual and Q-Q plots.

---

## Part B — Classification: Pima Indians Diabetes (or UCI Wine Quality)

### Dataset
Use one of the following binary classification datasets:
- **Pima Indians Diabetes** (download from UCI or Kaggle) — target: has diabetes yes/no.
- **Wine Quality** (sklearn or UCI) — binarize as good (score ≥ 7) vs not good.

### Tasks

#### B1: Data Exploration and Preprocessing
1. Load the dataset, explore shape, class distribution, summary statistics, correlation matrix.
2. Check for missing values. Impute or drop as appropriate.
3. Split train (70%) / test (30%) with stratification.

#### B2: From-Scratch and Sklearn
1. Implement binary logistic regression from scratch with cross-entropy + gradient descent (Section 6).
2. Fit `sklearn.linear_model.LogisticRegression` (with large `C` for unregularized).
3. Report accuracy, precision, recall, F1, and AUC on the held-out test set.

#### B3: Statsmodels Inference
1. Fit `sm.Logit(y, add_constant(X))` (or `smf.logit` with a formula).
2. Interpret the summary:
   - Which features are statistically significant?
   - Report the **odds ratio** ($e^{\hat{w}_j}$) and 95% CI for each feature. Is a 10% increase in `X_j` clinically meaningful?
   - Report McFadden pseudo-R² and the LLR p-value.

#### B4: Regularization
1. Fit L2, L1, and Elastic Net logistic regressions (pay attention to solver compatibility).
2. Plot coefficient paths for L1 and L2 as a function of `C` on a log scale.
3. How many features does L1 keep at the best `C`?
4. Replicate with `statsmodels.Logit.fit_regularized` and compare.

#### B5: Cross-Validation and Hyperparameter Tuning
1. Use `StratifiedKFold` (5 folds).
2. Use `LogisticRegressionCV` with `Cs = np.logspace(-3, 3, 20)` for both L1 and L2.
3. Use `GridSearchCV` to jointly tune `penalty ∈ {l1, l2, elasticnet}`, `C`, and `l1_ratio` (use `saga`).
4. Use `RandomizedSearchCV` with `loguniform(1e-3, 1e3)` for `C` and `uniform(0,1)` for `l1_ratio`.
5. Compare best CV scores, runtimes, and visualize.

#### B6: Final Model and Analysis
1. Refit the best model on the full training set.
2. Evaluate on the held-out test set: accuracy, precision, recall, F1, AUC.
3. Plot ROC curve, precision-recall curve, and confusion matrix.
4. **Bonus**:
   - Which features ended up being selected / shrunk to zero? Does that match domain intuition?
   - Threshold the predicted probability at something other than 0.5 (e.g. 0.3) and observe the precision–recall trade-off.
   - Compare your results with KNN / SVM classification on the same data.
   - If your dataset is imbalanced, re-run with `class_weight='balanced'` and discuss the effect on recall for the minority class.
